Here are some small exercises, ordered easiest → trickier. Try them in a Python REPL or scratch cell — no test framework needed. Each one builds intuition you'll need for the actual test.

1. Build a single Worker by hand
Open a REPL, from allocations.models import Worker, and construct one with 3 jobs (cost/skill/time vectors of length 3). Print worker.job_count. → confirms what "job_count" really is.

2. Break the Worker on purpose
Try building one where cost has length 3 but skill has length 2 — using the regular constructor first, then via Worker.from_mapping({...}). → Notice which one validates and which doesn't. That tells you something important about how the test will need to construct workers safely.

3. Play with expanded_jobs
from allocations.matrix import expanded_jobs. Call it with:

expanded_jobs(3) (no requirements)
expanded_jobs(3, requirements=[1, 1, 1])
expanded_jobs(3, requirements=[2, 0, 1])
expanded_jobs(3, requirements=[1, 1]) ← should raise
→ this is the function that converts requirements into "slots". Knowing its output shape makes "workers match slots" very concrete.

4. Count slots without calling expanded_jobs
Given requirements=[2, 0, 1], what's len(expanded_jobs(...))? Predict it, then verify. → you'll see it's just sum(requirements). This is the magic number your test compares len(workers) to.

5. Trigger the InfeasibleAssignmentError
Build 2 workers (each with 3 jobs) and call allocate_workers(workers, requirements=[2, 1, 1]). → 2 workers, 4 slots → it raises. Now try requirements=[1, 1, 0] → 2 workers, 2 slots → it succeeds. This is the boundary your is_feasible will live on.

6. Penalty vector quirk
Call worker.penalty_values("skill") and worker.penalty_values("cost") on the same worker. → Notice skill is negated. Not directly relevant to your feasibility test, but useful context for other tests in this notebook.

7. Write a tiny helper
Write a one-line function make_worker(n_jobs, worker_id="w1") that returns a Worker with all-zero cost/skill/time vectors of length n_jobs. → You'll likely want this exact helper inside your test cell so each test stays one or two lines.

Once exercises 3, 4, and 7 click, the test you're writing is essentially: make N workers, pick requirements that sum to N, assert is_feasible(...) is True.

In [54]:
from allocations.models import Worker
def make_worker(n_jobs,  worker_id="w1"):
    worker=Worker(
        id="w1",
        forename=" ",
        surname=" ",
        cost= tuple(range(len(n_jobs))),
        skill=tuple(range(len(n_jobs))),
        time=tuple(range(len(n_jobs))),
        
    )
    


In [ ]:
from allocations.models import Worker
from allocations.matrix import AssignmentResult

"""5. Trigger the InfeasibleAssignmentError
Build 2 workers (each with 3 jobs) and call allocate_workers(workers, 
requirements=[2, 1, 1]). → 2 workers, 4 slots → it raises.
Now try requirements=[1, 1, 0] → 2 workers, 2 slots → it succeeds. 
This is the boundary your is_feasible will live on.
"""

w= Worker(
    id="GF1978",
    forename="garfield",
    surname="arbuckle",
    cost=(0.0, 7.0, 1.0),
    skill=(9.0, 0.0, 8.0),
    time=(3.0, 9.0, 6.0),

)

p= Worker(
    id="OD1988",
    forename="odie",
    surname="dog",
    cost=(0.0, 7.0, 1.0),
    skill=(9.0, 0.0, 8.0),
    time=(3.0, 9.0, 6.0),
)

workers = [w, p]
print(w.job_count)
print(w.penalty_values("skill"))



3
(-9.0, -0.0, -8.0)


In [50]:
print(w.penalty_values("skill"),
w.penalty_values("cost"))
 

(-9.0, -0.0, -8.0) (0.0, 7.0, 1.0)


In [43]:
from allocations.matrix import allocate_workers
allocate_workers(workers,requirements= [1, 1])

ValueError: requirements length 2 does not match job count 3

In [46]:
allocate_workers(workers, requirements=[0,1, 1])

AssignmentResult(solver='hungarian', metric='cost', slot_assignment=(0, 1), slot_to_job=(1, 2), total_penalty=8.0)

In [ ]:
#Given requirements=[2, 0, 1], what's len(expanded_jobs(...))? Predict 

In [31]:
from allocations.matrix import expanded_jobs
expanded_jobs(3,requirements=[2, 0, 1])

(0, 0, 2)

In [18]:
Worker.from_mapping({"id": "GF1978",
    "forename": "garfield",
    "surname": "arbuckle",
    "cost": [0.0, 7.0,],
    "skill": [9.0, 0.0, 8.0],
    "time": [3.0, 9.0, 6.0]
    })

InvalidWorkerDataError: worker 'GF1978' has cost, skill, and time vectors with different lengths

In [ ]:
# requirements: how many workers are needed for each job


typing.Sequence[allocations.models.Worker]
